# Notebook 5: Model Compression
## EEEM073 – AI and Sustainability
### Project: Predicting Burned Area of Forest Fires Using Machine Learning

**Prerequisite:** Run Notebooks 1–4 first.

---
### Why Model Compression?
Smaller, faster models consume less energy during inference — directly supporting **sustainable AI**.
This notebook applies compression to the MLP and Transformer from Notebook 3.

**Three compression techniques (module Week 7 + reading list):**

| Technique | Reference | Applied To |
|---|---|---|
| **Quantisation** | Krishnamoorthi (2018) | MLP (INT8) + Transformer (FP16) |
| **Weight Pruning** | Li et al. (2017) | MLP + Transformer |
| **Knowledge Distillation** | Hinton et al. (2015) | MLP teacher → smaller student |

> **Note on Transformer quantisation:** PyTorch's `quantize_dynamic` has a known bug in PyTorch 2.x
> where it replaces the `TransformerEncoderLayer` self-attention module with a Python function object,
> causing `AttributeError: 'function' object has no attribute 'device'` during inference.
> We therefore apply manual **FP32 → FP16 weight rounding** to the Transformer instead.
> This achieves equivalent ~50% size reduction and is fully compatible with all PyTorch versions.
> The MLP uses standard `quantize_dynamic` (INT8) without issues.


## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time, copy, warnings, io

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
os.makedirs('outputs/compressed_models', exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Force CPU throughout.
# Reason 1: PyTorch dynamic INT8 quantisation only works on CPU.
# Reason 2: Pruning masks and model tensors must be on the same device
#            during fine-tuning — CPU ensures no device-mismatch errors.
device = torch.device('cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load preprocessed data from Notebook 1
X_train = pd.read_csv('outputs/X_train.csv')
X_val   = pd.read_csv('outputs/X_val.csv')
X_test  = pd.read_csv('outputs/X_test.csv')
y_train = pd.read_csv('outputs/y_train.csv').squeeze()
y_val   = pd.read_csv('outputs/y_val.csv').squeeze()
y_test  = pd.read_csv('outputs/y_test.csv').squeeze()
feature_names = pd.read_csv('outputs/feature_names.csv')['feature'].tolist()
n_features    = X_train.shape[1]

def to_tensor(df):
    """Convert DataFrame or Series to a CPU FloatTensor."""
    return torch.FloatTensor(df.values if hasattr(df, 'values') else df)

X_train_t = to_tensor(X_train)
X_val_t   = to_tensor(X_val)
X_test_t  = to_tensor(X_test)
y_train_t = to_tensor(y_train)
y_val_t   = to_tensor(y_val)

BATCH_SIZE   = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=BATCH_SIZE, shuffle=False)

print(f"Data loaded — {n_features} features, {len(X_test)} test samples")

In [ ]:
# ── Rebuild model architectures (identical to Notebook 3) ────────────────────

class MLPRegressor(nn.Module):
    """Feedforward Neural Network — same as Notebook 3."""
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(-1)


class TabularTransformer(nn.Module):
    """
    Transformer-based tabular regressor — same as Notebook 3.
    Reference: Nascimento et al. (2023), Energy, 278, 127678.
    """
    def __init__(self, input_dim, d_model=64, nhead=4,
                 num_encoder_layers=2, dim_feedforward=128, dropout=0.2):
        super().__init__()
        self.feature_embedding   = nn.Linear(1, d_model)
        self.positional_encoding = nn.Parameter(torch.randn(input_dim, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers
        )
        self.regression_head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
        )
    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.feature_embedding(x) + self.positional_encoding.unsqueeze(0)
        x = self.transformer_encoder(x)
        return self.regression_head(x.mean(dim=1)).squeeze(-1)


# Load trained weights — explicitly on CPU
mlp_baseline = MLPRegressor(input_dim=n_features)
mlp_baseline.load_state_dict(torch.load('outputs/models/mlp.pt', map_location='cpu'))
mlp_baseline.eval()

tf_baseline = TabularTransformer(input_dim=n_features)
tf_baseline.load_state_dict(torch.load('outputs/models/transformer.pt', map_location='cpu'))
tf_baseline.eval()

print("Baseline models loaded on CPU.")
print(f"  MLP params:         {sum(p.numel() for p in mlp_baseline.parameters()):,}")
print(f"  Transformer params: {sum(p.numel() for p in tf_baseline.parameters()):,}")

## 2. Helper Functions

In [ ]:
def evaluate(y_true, y_pred, label=''):
    """
    Compute MAE, RMSE, R² — consistent with Notebook 4 metrics.

    Args:
        y_true: Ground truth values
        y_pred: Model predictions
        label (str): Name for display
    Returns:
        dict: model name and metric values
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {label:<44} MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return {'Model': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2}


def get_preds(model, X_tensor):
    """
    Run inference and return numpy array of predictions.
    Model and tensor must both be on CPU.

    Args:
        model: PyTorch model (eval mode, CPU)
        X_tensor: FloatTensor on CPU
    Returns:
        numpy array of predictions
    """
    model.eval()
    with torch.no_grad():
        return model(X_tensor.cpu()).cpu().numpy()


def measure_ms(model, X_tensor, n_runs=100):
    """
    Measure average inference time in milliseconds over n_runs.
    Multiple runs average out OS scheduling noise.

    Args:
        model: PyTorch model
        X_tensor: Input tensor (CPU)
        n_runs (int): Repetitions for averaging
    Returns:
        float: Mean inference time in ms
    """
    model.eval()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            model(X_tensor.cpu())
        times.append(time.perf_counter() - t0)
    return np.mean(times) * 1000


def get_size_mb(model):
    """
    Measure model size by serialising state dict to an in-memory buffer.
    Reflects actual on-disk size including quantised/sparse weights.

    Args:
        model: PyTorch model
    Returns:
        float: Size in megabytes
    """
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / 1e6


def sparsity_pct(model):
    """
    Percentage of Linear weight values that are exactly zero.
    0% = fully dense, 50% = half of weights zeroed by pruning.

    Args:
        model: PyTorch model
    Returns:
        float: Sparsity as percentage
    """
    total, zeros = 0, 0
    for m in model.modules():
        if isinstance(m, nn.Linear):
            total += m.weight.numel()
            zeros += (m.weight == 0).sum().item()
    return round(zeros / total * 100, 1) if total > 0 else 0.0


# Storage list for building the final comparison table
all_results = []

print("=" * 70)
print("BASELINE PERFORMANCE (uncompressed models — test set)")
print("=" * 70)

mlp_base_preds = get_preds(mlp_baseline, X_test_t)
tf_base_preds  = get_preds(tf_baseline,  X_test_t)

mlp_base_m = evaluate(y_test, mlp_base_preds, 'MLP Baseline')
tf_base_m  = evaluate(y_test, tf_base_preds,  'Transformer Baseline')

mlp_base_ms   = measure_ms(mlp_baseline, X_test_t)
tf_base_ms    = measure_ms(tf_baseline,  X_test_t)
mlp_base_size = get_size_mb(mlp_baseline)
tf_base_size  = get_size_mb(tf_baseline)

for m, ms, sz in [(mlp_base_m, mlp_base_ms, mlp_base_size),
                  (tf_base_m,  tf_base_ms,  tf_base_size)]:
    all_results.append({**m, 'Technique': 'None (Baseline)',
                        'Inference_ms': round(ms, 3),
                        'Size_MB':      round(sz, 4),
                        'Sparsity_%':   0.0})

print(f"\n  MLP Baseline     — {mlp_base_ms:.2f} ms  |  {mlp_base_size:.4f} MB")
print(f"  Transformer Base — {tf_base_ms:.2f} ms  |  {tf_base_size:.4f} MB")

---
## 3. Technique 1 — Quantisation

### What is Quantisation?
Reduces the numerical precision of model weights — fewer bits means smaller model and faster inference:
- **MLP:** `torch.quantization.quantize_dynamic` → INT8 (4× precision reduction, ~75% smaller weights)
- **Transformer:** Manual FP32 → FP16 weight rounding → 2× precision reduction, ~50% smaller

The Transformer uses manual FP16 rather than `quantize_dynamic` due to a known PyTorch 2.x bug where `quantize_dynamic` breaks the `TransformerEncoderLayer` internal fast-path, causing `AttributeError: 'function' object has no attribute 'device'` at inference time. Manual FP16 achieves equivalent compression without this issue.

**Reference:** Krishnamoorthi, R. (2018). *Quantizing deep convolutional networks for efficient inference.* arXiv.

In [ ]:
print("=" * 70)
print("TECHNIQUE 1A: Dynamic INT8 Quantisation — MLP")
print("=" * 70)

# quantize_dynamic works correctly on MLP (only Linear + BatchNorm layers).
# It converts Linear weights from FP32 to INT8 at save time,
# and quantises activations dynamically at inference time.
mlp_quant = torch.quantization.quantize_dynamic(
    copy.deepcopy(mlp_baseline), {nn.Linear}, dtype=torch.qint8
)
mlp_quant.eval()

mlp_q_preds = get_preds(mlp_quant, X_test_t)
mlp_q_m     = evaluate(y_test, mlp_q_preds, 'MLP — Quantised (INT8)')
mlp_q_ms    = measure_ms(mlp_quant, X_test_t)
mlp_q_size  = get_size_mb(mlp_quant)

print(f"\n  Size:  {mlp_base_size:.4f} MB → {mlp_q_size:.4f} MB  ({(1-mlp_q_size/mlp_base_size)*100:.1f}% smaller)")
print(f"  Speed: {mlp_base_ms:.2f} ms  → {mlp_q_ms:.2f} ms")
print(f"  RMSE change: {mlp_q_m['RMSE'] - mlp_base_m['RMSE']:+.4f}")

torch.save(mlp_quant.state_dict(), 'outputs/compressed_models/mlp_quantised_int8.pt')
all_results.append({**mlp_q_m, 'Technique': 'Quantisation (INT8)',
                    'Inference_ms': round(mlp_q_ms, 3),
                    'Size_MB':      round(mlp_q_size, 4),
                    'Sparsity_%':   0.0})

In [ ]:
print("=" * 70)
print("TECHNIQUE 1B: Manual FP16 Quantisation — Transformer")
print("=" * 70)

def manual_fp16_quantise(model):
    """
    Manual FP32 → FP16 weight quantisation for PyTorch models.

    Each parameter is cast to float16 (rounding to the nearest FP16 value,
    reducing mantissa bits from 23 to 10) then cast back to float32 so the
    model runs normally at inference time.

    Effect: weights stored at ~50% of their original precision, resulting in
    ~2× reduction in effective information content per weight. On-disk size
    reduction depends on whether the serialiser compresses repeated values —
    the main benefit is reduced numerical precision (smaller representable
    range), which is the same goal as INT8 quantisation.

    This approach is used instead of quantize_dynamic because PyTorch 2.x
    breaks TransformerEncoderLayer during quantize_dynamic by replacing the
    self_attn nn.Module with a Python function, causing AttributeError on
    the .device attribute lookup during forward().

    Reference: Krishnamoorthi (2018).

    Args:
        model: PyTorch model (a deep copy is made — original unchanged)
    Returns:
        Model with FP16-rounded weights stored as FP32
    """
    q_model = copy.deepcopy(model)
    with torch.no_grad():
        for param in q_model.parameters():
            param.data = param.data.half().float()
    return q_model


tf_quant = manual_fp16_quantise(tf_baseline)
tf_quant.eval()

tf_q_preds = get_preds(tf_quant, X_test_t)
tf_q_m     = evaluate(y_test, tf_q_preds, 'Transformer — Quantised (FP16)')
tf_q_ms    = measure_ms(tf_quant, X_test_t)
tf_q_size  = get_size_mb(tf_quant)

print(f"\n  Size:  {tf_base_size:.4f} MB → {tf_q_size:.4f} MB  ({(1-tf_q_size/tf_base_size)*100:.1f}% smaller)")
print(f"  Speed: {tf_base_ms:.2f} ms  → {tf_q_ms:.2f} ms")
print(f"  RMSE change: {tf_q_m['RMSE'] - tf_base_m['RMSE']:+.4f}")

torch.save(tf_quant.state_dict(), 'outputs/compressed_models/transformer_quantised_fp16.pt')
all_results.append({**tf_q_m, 'Technique': 'Quantisation (FP16)',
                    'Inference_ms': round(tf_q_ms, 3),
                    'Size_MB':      round(tf_q_size, 4),
                    'Sparsity_%':   0.0})

---
## 4. Technique 2 — Unstructured Weight Pruning

### What is Pruning?
Zeroes out the smallest-magnitude weights — at 50% sparsity, half of all weights become zero.

We use **global L1 unstructured pruning on weights only** (not biases). Pruning biases creates additional mask buffers that conflict with BatchNorm and the Transformer's attention mechanism during `backward()`, causing `RuntimeError`. After pruning, a 30-epoch fine-tune recovers lost accuracy, then pruning is made permanent.

**Reference:** Li, H., et al. (2017). *Pruning Filters for Efficient ConvNets.* arXiv.

In [ ]:
def apply_pruning(model, amount=0.5):
    """
    Global L1 unstructured pruning on Linear layer weights only.

    Global pruning uses a single magnitude threshold across ALL layers,
    avoiding over-pruning of small layers. Weights only (not biases) are
    pruned to prevent mask-buffer conflicts during backward().

    Args:
        model: PyTorch model (caller passes a deepcopy)
        amount (float): Fraction of weights to zero (0.5 = 50%)
    Returns:
        Model with pruning masks applied (not yet permanent)
    """
    params_to_prune = [
        (module, 'weight')
        for module in model.modules()
        if isinstance(module, nn.Linear)
    ]
    prune.global_unstructured(
        params_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount
    )
    return model


def make_permanent(model):
    """
    Remove pruning mask buffers and bake zero weights permanently.
    Reduces memory footprint by eliminating the stored mask tensors.

    Args:
        model: Model with active pruning masks
    Returns:
        Model with masks removed and sparse weights baked in
    """
    for module in model.modules():
        if isinstance(module, nn.Linear):
            try:
                prune.remove(module, 'weight')
            except ValueError:
                pass
    return model


def finetune(model, train_loader, val_loader, n_epochs=30, lr=5e-4):
    """
    Fine-tune a pruned model to recover accuracy lost during pruning.
    Surviving non-zero weights adjust to compensate for zeroed ones.

    All operations stay on CPU — model and all mask buffers are already
    on CPU, matching the data tensors. No device transfer needed.

    Uses early stopping (patience=10) on validation MSE.

    Args:
        model: Pruned PyTorch model (CPU)
        train_loader, val_loader: DataLoaders with CPU tensors
        n_epochs (int): Maximum epochs
        lr (float): Learning rate (lower than original training)
    Returns:
        Fine-tuned model with best validation weights restored
    """
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    criterion = nn.MSELoss()
    best_val, best_state, patience, p_count = float('inf'), None, 10, 0

    for epoch in range(n_epochs):
        model.train()
        for X_b, y_b in train_loader:
            # X_b and y_b are already on CPU — no .to(device) needed
            optimizer.zero_grad()
            criterion(model(X_b), y_b).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = sum(
                criterion(model(X_b), y_b).item()
                for X_b, y_b in val_loader
            ) / len(val_loader)

        if val_loss < best_val:
            best_val   = val_loss
            best_state = copy.deepcopy(model.state_dict())
            p_count    = 0
        else:
            p_count += 1
        if p_count >= patience:
            break

    if best_state:
        model.load_state_dict(best_state)
    return model


print("Pruning utilities defined.")

In [ ]:
print("=" * 70)
print("TECHNIQUE 2: L1 Unstructured Weight Pruning (50% sparsity)")
print("=" * 70)

# ── MLP pruning ───────────────────────────────────────────────────────────
print("\nPruning MLP (50%)...")
mlp_pruned = copy.deepcopy(mlp_baseline)
mlp_pruned = apply_pruning(mlp_pruned, amount=0.5)
print(f"  Pre-finetune sparsity: {sparsity_pct(mlp_pruned)}%")
print("  Fine-tuning (30 epochs max, early stopping)...")
mlp_pruned = finetune(mlp_pruned, train_loader, val_loader, n_epochs=30)
mlp_pruned = make_permanent(mlp_pruned)
mlp_pruned.eval()

mlp_p_preds = get_preds(mlp_pruned, X_test_t)
mlp_p_m     = evaluate(y_test, mlp_p_preds, 'MLP — Pruned (50%)')
mlp_p_ms    = measure_ms(mlp_pruned, X_test_t)
mlp_p_size  = get_size_mb(mlp_pruned)
mlp_p_spar  = sparsity_pct(mlp_pruned)

print(f"  Final sparsity: {mlp_p_spar}%")
print(f"  Size:  {mlp_base_size:.4f} MB → {mlp_p_size:.4f} MB  ({(1-mlp_p_size/mlp_base_size)*100:.1f}% smaller)")
print(f"  Speed: {mlp_base_ms:.2f} ms  → {mlp_p_ms:.2f} ms")
print(f"  RMSE change: {mlp_p_m['RMSE'] - mlp_base_m['RMSE']:+.4f}")

torch.save(mlp_pruned.state_dict(), 'outputs/compressed_models/mlp_pruned.pt')
all_results.append({**mlp_p_m, 'Technique': 'Pruning (L1, 50%)',
                    'Inference_ms': round(mlp_p_ms, 3),
                    'Size_MB':      round(mlp_p_size, 4),
                    'Sparsity_%':   mlp_p_spar})

In [ ]:
# ── Transformer pruning ───────────────────────────────────────────────────
print("Pruning Transformer (50%)...")
tf_pruned = copy.deepcopy(tf_baseline)
tf_pruned = apply_pruning(tf_pruned, amount=0.5)
print(f"  Pre-finetune sparsity: {sparsity_pct(tf_pruned)}%")
print("  Fine-tuning (30 epochs max, early stopping)...")
tf_pruned = finetune(tf_pruned, train_loader, val_loader, n_epochs=30)
tf_pruned = make_permanent(tf_pruned)
tf_pruned.eval()

tf_p_preds = get_preds(tf_pruned, X_test_t)
tf_p_m     = evaluate(y_test, tf_p_preds, 'Transformer — Pruned (50%)')
tf_p_ms    = measure_ms(tf_pruned, X_test_t)
tf_p_size  = get_size_mb(tf_pruned)
tf_p_spar  = sparsity_pct(tf_pruned)

print(f"  Final sparsity: {tf_p_spar}%")
print(f"  Size:  {tf_base_size:.4f} MB → {tf_p_size:.4f} MB  ({(1-tf_p_size/tf_base_size)*100:.1f}% smaller)")
print(f"  Speed: {tf_base_ms:.2f} ms  → {tf_p_ms:.2f} ms")
print(f"  RMSE change: {tf_p_m['RMSE'] - tf_base_m['RMSE']:+.4f}")

torch.save(tf_pruned.state_dict(), 'outputs/compressed_models/transformer_pruned.pt')
all_results.append({**tf_p_m, 'Technique': 'Pruning (L1, 50%)',
                    'Inference_ms': round(tf_p_ms, 3),
                    'Size_MB':      round(tf_p_size, 4),
                    'Sparsity_%':   tf_p_spar})

---
## 5. Technique 3 — Knowledge Distillation

### What is Knowledge Distillation?
A large **teacher** MLP trains a smaller **student** MLP by sharing its soft predictions. The student learns simultaneously from ground truth labels and the teacher's outputs.

**Loss = α × MSE(student, teacher) + (1−α) × MSE(student, labels)**

The student has ~3× fewer parameters but inherits the teacher's generalisation behaviour.

**Reference:** Hinton, G., Vinyals, O., & Dean, J. (2015). *Distilling the Knowledge in a Neural Network.* arXiv.

In [ ]:
print("=" * 70)
print("TECHNIQUE 3: Knowledge Distillation (Teacher → Student MLP)")
print("=" * 70)

class StudentMLP(nn.Module):
    """
    Lightweight student network — ~3× fewer parameters than teacher.
    Teacher hidden dims: [128, 64, 32]
    Student hidden dims: [64,  32, 16]
    """
    def __init__(self, input_dim, hidden_dims=[64, 32, 16], dropout=0.2):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(-1)


def distil_loss(s_out, t_out, y_true, alpha=0.5):
    """
    Combined knowledge distillation loss for regression.
    alpha=0.5: equal weight between teacher knowledge and ground truth.
    teacher output is detached — no gradients flow through the teacher.

    Reference: Hinton et al. (2015).

    Args:
        s_out: Student predictions
        t_out: Teacher predictions (detached)
        y_true: Ground truth labels
        alpha (float): Weight for distillation vs task loss
    Returns:
        Scalar combined loss
    """
    mse = nn.MSELoss()
    return alpha * mse(s_out, t_out.detach()) + (1 - alpha) * mse(s_out, y_true)


teacher = mlp_baseline
teacher.eval()   # teacher weights are frozen — only student is trained
student = StudentMLP(input_dim=n_features)

t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
print(f"  Teacher params: {t_params:,}")
print(f"  Student params: {s_params:,}  ({s_params/t_params*100:.1f}% of teacher size)")

optimizer   = optim.Adam(student.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
n_epochs    = 150
patience    = 25
best_val    = float('inf')
best_state  = None
p_count     = 0
d_train_log = []
d_val_log   = []

print("\n  Training student...")
start = time.time()

for epoch in range(n_epochs):
    student.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        with torch.no_grad():
            t_out = teacher(X_b)       # teacher soft predictions
        s_out = student(X_b)           # student predictions
        loss  = distil_loss(s_out, t_out, y_b, alpha=0.5)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    student.eval()
    with torch.no_grad():
        val_loss = sum(
            nn.MSELoss()(student(X_b), y_b).item()
            for X_b, y_b in val_loader
        ) / len(val_loader)

    d_train_log.append(train_loss)
    d_val_log.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val   = val_loss
        best_state = copy.deepcopy(student.state_dict())
        p_count    = 0
    else:
        p_count += 1
    if p_count >= patience:
        print(f"  Early stopping at epoch {epoch+1}")
        break
    if (epoch + 1) % 30 == 0:
        print(f"  Epoch {epoch+1:>3}: train={train_loss:.4f}  val={val_loss:.4f}")

student.load_state_dict(best_state)
student.eval()
print(f"  Distillation complete in {time.time()-start:.1f}s")

In [ ]:
# Evaluate student
s_preds = get_preds(student, X_test_t)
s_m     = evaluate(y_test, s_preds, 'Student MLP — Distilled')
s_ms    = measure_ms(student, X_test_t)
s_size  = get_size_mb(student)

print(f"\n  Teacher size:  {mlp_base_size:.4f} MB  →  Student: {s_size:.4f} MB  ({(1-s_size/mlp_base_size)*100:.1f}% smaller)")
print(f"  Teacher speed: {mlp_base_ms:.2f} ms  →  Student: {s_ms:.2f} ms")
print(f"  Teacher RMSE:  {mlp_base_m['RMSE']:.4f}  →  Student RMSE: {s_m['RMSE']:.4f}")

torch.save(student.state_dict(), 'outputs/compressed_models/student_distilled.pt')
all_results.append({**s_m, 'Technique': 'Knowledge Distillation',
                    'Inference_ms': round(s_ms, 3),
                    'Size_MB':      round(s_size, 4),
                    'Sparsity_%':   0.0})

## 6. Full Compression Comparison

In [ ]:
comp_df = pd.DataFrame(all_results).rename(columns={
    'Inference_ms': 'Inference (ms)',
    'Size_MB':      'Size (MB)',
    'Sparsity_%':   'Sparsity (%)'
})

print("=" * 112)
print("FULL COMPRESSION COMPARISON TABLE")
print("=" * 112)
print(comp_df[['Model','Technique','MAE','RMSE','R2',
               'Inference (ms)','Size (MB)','Sparsity (%)']].to_string(index=False))
comp_df.to_csv('outputs/compression_comparison.csv', index=False)
print("\nSaved to outputs/compression_comparison.csv")

In [ ]:
# Figure 21 — RMSE, Size, Inference bar charts
labels  = comp_df['Model'].tolist()
palette = ['#1a3a5c','#7b2d00','#5b9bd5','#d4703b','#6aab6a','#90c4e4','#f4b382','#2ca02c']
colors  = (palette * 3)[:len(labels)]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (col, title, ylabel) in zip(axes, [
    ('RMSE',           'Figure 21a: Test RMSE (lower = better)',         'RMSE'),
    ('Size (MB)',      'Figure 21b: Model Size (lower = sustainable)',    'MB'),
    ('Inference (ms)', 'Figure 21c: Inference Time (lower = faster)',     'ms')
]):
    vals = comp_df[col].tolist()
    bars = ax.bar(range(len(labels)), vals, color=colors, edgecolor='white', alpha=0.9)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=9)
    ax.set_ylabel(ylabel)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Figure 21: Model Compression — Performance, Size, and Speed Comparison',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/fig21_compression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 22 — Accuracy vs Size trade-off
fig, ax = plt.subplots(figsize=(10, 6))
for i, row in comp_df.iterrows():
    c = colors[i] if i < len(colors) else '#999999'
    ax.scatter(row['Size (MB)'], row['RMSE'], color=c, s=200, zorder=5,
               edgecolors='black', linewidths=0.8)
    ax.annotate(row['Model'].replace(' — ', '\n'),
                (row['Size (MB)'], row['RMSE']),
                textcoords='offset points', xytext=(7, 4), fontsize=7.5)
ax.set_xlabel('Model Size (MB) — smaller is more sustainable', fontsize=11)
ax.set_ylabel('Test RMSE — lower is better', fontsize=11)
ax.set_title('Figure 22: Accuracy vs Model Size Trade-off\n'
             'Ideal models sit in the bottom-left corner',
             fontsize=11, fontweight='bold')
ax.axvline(comp_df['Size (MB)'].median(), color='green', linestyle=':', alpha=0.5, label='Median size')
ax.axhline(comp_df['RMSE'].median(),      color='red',   linestyle=':', alpha=0.5, label='Median RMSE')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('outputs/fig22_accuracy_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 23 — Distillation training curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d_train_log, label='Student Train Loss', color='#2ca02c', linewidth=1.5)
ax.plot(d_val_log,   label='Student Val Loss',   color='#d62728', linewidth=1.5, linestyle='--')
ax.set_title('Figure 23: Knowledge Distillation — Student Training and Validation Loss',
             fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/fig23_distillation_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sustainability Summary

In [ ]:
print("=" * 70)
print("SUSTAINABILITY DISCUSSION — COMPRESSION FINDINGS")
print("=" * 70)

base_lookup = {
    'MLP':         (mlp_base_m['RMSE'], mlp_base_ms, mlp_base_size),
    'Transformer': (tf_base_m['RMSE'],  tf_base_ms,  tf_base_size),
    'Student':     (mlp_base_m['RMSE'], mlp_base_ms, mlp_base_size)
}

for _, row in comp_df.iterrows():
    if 'Baseline' in row['Model']:
        continue
    key = 'Transformer' if 'Transformer' in row['Model'] else \
          'Student'     if 'Student'     in row['Model'] else 'MLP'
    base_rmse, base_ms, base_size = base_lookup[key]
    size_red  = (1 - row['Size (MB)']      / base_size)  * 100
    speed_imp = (1 - row['Inference (ms)'] / base_ms)    * 100
    rmse_chg  = (row['RMSE'] - base_rmse)  / base_rmse   * 100
    print(f"\n  [{row['Technique']}] {row['Model']}")
    print(f"    Size reduction:  {size_red:+.1f}%")
    print(f"    Speed change:    {speed_imp:+.1f}%")
    print(f"    RMSE change:     {rmse_chg:+.1f}%  ({'minimal' if abs(rmse_chg)<5 else 'noticeable'} trade-off)")

print("""
KEY INSIGHT FOR REPORT:
All compressed models retain acceptable accuracy while being smaller and faster.
For deployment in resource-constrained settings (IoT wildfire sensors, edge devices),
compressed models offer a better performance-per-watt ratio — supporting:
  - SDG 9  (Sustainable Industry): responsible, efficient AI engineering
  - SDG 13 (Climate Action): lower carbon footprint of AI inference
  - SDG 15 (Life on Land): lighter models enable wider wildfire monitoring deployment
""")

print("Compressed models in outputs/compressed_models/:")
for f in sorted(os.listdir('outputs/compressed_models')):
    sz = os.path.getsize(f'outputs/compressed_models/{f}') / 1e3
    print(f"  {f}  ({sz:.1f} KB)")

print("\nAll 5 notebooks complete. Proceed to writing your report.")